In [ ]:
! pip install /kaggle/input/litmodels-bolts/lightning_bolts-0.3.3-py3-none-any.whl

In [ ]:
!pip install /kaggle/input/kerasapplications -q
!pip install /kaggle/input/efficientnet-keras-source-code/ -q --no-deps

In [ ]:
!conda install '/kaggle/input/pydicom-conda-helper/libjpeg-turbo-2.1.0-h7f98852_0.tar.bz2' -c conda-forge -y
!conda install '/kaggle/input/pydicom-conda-helper/libgcc-ng-9.3.0-h2828fa1_19.tar.bz2' -c conda-forge -y
!conda install '/kaggle/input/pydicom-conda-helper/gdcm-2.8.9-py37h500ead1_1.tar.bz2' -c conda-forge -y
!conda install '/kaggle/input/pydicom-conda-helper/conda-4.10.1-py37h89c1867_0.tar.bz2' -c conda-forge -y
!conda install '/kaggle/input/pydicom-conda-helper/certifi-2020.12.5-py37h89c1867_1.tar.bz2' -c conda-forge -y
!conda install '/kaggle/input/pydicom-conda-helper/openssl-1.1.1k-h7f98852_0.tar.bz2' -c conda-forge -y

# References

- https://www.kaggle.com/illidan7/siim-covid-19-classification-train
- https://www.kaggle.com/illidan7/litmodelscolab-evaluation/data
- https://www.kaggle.com/shivanandmn/efficientnet-pytorch-lightning-train-inference
- https://www.kaggle.com/illidan7/siim-covid-19-frankenstein/notebook
- https://www.kaggle.com/whitegg/inference-studyclass-baseline

# Load libraries

In [ ]:
import pytorch_lightning as pl
from pytorch_lightning.metrics.functional import accuracy
from pytorch_lightning.callbacks import ModelCheckpoint

In [ ]:
import platform
import numpy as np 
import pandas as pd 
import os
from tqdm.notebook import tqdm
import cv2
import pydicom
import gdcm
import glob
import gc
from math import ceil
import matplotlib.pyplot as plt
from pydicom.pixel_data_handlers.util import apply_voi_lut
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

import albumentations

# from sklearn.model_selection import StratifiedKFold, GroupKFold

import math
import psutil

DATA_PATH = '/kaggle/input/siim-covid19-detection/'

import sys
sys.path.append('../input/timm-pytorch-image-models/pytorch-image-models-master')
sys.path.append('../usr/lib/siim_covid19_littrainers_py/')

import timm
import siim_covid19_littrainers_py as lit

In [ ]:
from PIL import Image

import torch
torch.manual_seed(0)
torch.backends.cudnn.deterministic = False
torch.backends.cudnn.benchmark = True

# import timm

import torchvision.models as models
import torchvision.transforms as transforms
import torchvision.datasets as datasets
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.autograd import Variable
from torch.cuda.amp import GradScaler, autocast
from torch.utils.data import Dataset, DataLoader
# from pytorch_metric_learning import losses
from torch.optim import lr_scheduler

import warnings
warnings.simplefilter('ignore')

In [ ]:
df = pd.read_csv('/kaggle/input/siim-covid19-detection/sample_submission.csv')
if df.shape[0] == 2477:
    fast_sub = True
    fast_df = pd.DataFrame(([['00086460a852_study', 'negative 1 0 0 1 1'], 
                         ['000c9c05fd14_study', 'negative 1 0 0 1 1'], 
                         ['65761e66de9f_image', 'none 1 0 0 1 1'], 
                         ['51759b5579bc_image', 'none 1 0 0 1 1']]), 
                       columns=['id', 'PredictionString'])
else:
    fast_sub = False

# fast_sub = False

# Config

In [ ]:
class Config:
    train_pcent = 0.85
    model_name = 'resnet50'
    image_size = (400, 400)
    num_classes = 4
    batch_size = 32
    num_workers = 8
    NB_EPOCHS = 30
    scaler = GradScaler()
    scheduler = 'CosineAnnealingLR'
    weight_decay = 1e-6
    T_max = 10
    min_lr = 1e-6
    lr = 1e-5


# Dataset - Colab

In [ ]:
def dicom2array(path, voi_lut=True, fix_monochrome=True):
    dicom = pydicom.read_file(path)
    # VOI LUT (if available by DICOM device) is used to
    # transform raw DICOM data to "human-friendly" view
    if voi_lut:
        data = apply_voi_lut(dicom.pixel_array, dicom)
    else:
        data = dicom.pixel_array
    # depending on this value, X-ray may look inverted - fix that:
    if fix_monochrome and dicom.PhotometricInterpretation == "MONOCHROME1":
        data = np.amax(data) - data
    data = data - np.min(data)
    data = data / np.max(data)
    data = (data * 255).astype(np.uint8)
    return data

In [ ]:
class SIIMData(Dataset):
    def __init__(self, df, is_train=True, augments=None, img_size=Config.image_size):
        super().__init__()
        self.df = df.sample(frac=1).reset_index(drop=True)
        self.is_train = is_train
        self.augments = augments
        self.img_size = img_size
        
    def __getitem__(self, idx):
        image_id = self.df['StudyInstanceUID'].values[idx]
        
        image_path = self.df['path'].values[idx]
        image = dicom2array(image_path)
        image = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)
        image = cv2.resize(image, Config.image_size)
        
        # Augments must be albumentations
        if self.augments:
            image = self.augments(image=image)['image']
        else:
            image = torch.tensor(image)
        
        if self.is_train:
            label = self.df[self.df['StudyInstanceUID'] == image_id].values.tolist()[0][3:7].index(1)
            return image, torch.tensor(label)
        
        return image
    
    def __len__(self):
        return len(self.df)

# Dataset - Efnb7

In [ ]:
import numpy as np
import pydicom
from pydicom.pixel_data_handlers.util import apply_voi_lut

def read_xray(path, voi_lut = True, fix_monochrome = True):
    # Original from: https://www.kaggle.com/raddar/convert-dicom-to-np-array-the-correct-way
    dicom = pydicom.read_file(path)
    
    # VOI LUT (if available by DICOM device) is used to transform raw DICOM data to 
    # "human-friendly" view
    if voi_lut:
        data = apply_voi_lut(dicom.pixel_array, dicom)
    else:
        data = dicom.pixel_array
               
    # depending on this value, X-ray may look inverted - fix that:
    if fix_monochrome and dicom.PhotometricInterpretation == "MONOCHROME1":
        data = np.amax(data) - data
        
    data = data - np.min(data)
    data = data / np.max(data)
    data = (data * 255).astype(np.uint8)
        
    return data

In [ ]:
def resize(array, size, keep_ratio=False, resample=Image.LANCZOS):
    # Original from: https://www.kaggle.com/xhlulu/vinbigdata-process-and-resize-to-image
    im = Image.fromarray(array)
    
    if keep_ratio:
        im.thumbnail((size, size), resample)
    else:
        im = im.resize((size, size), resample)
    
    return im

In [ ]:
split = 'test'
save_dir = f'/kaggle/tmp/{split}/'

os.makedirs(save_dir, exist_ok=True)

save_dir = f'/kaggle/tmp/{split}/study/'
os.makedirs(save_dir, exist_ok=True)

if fast_sub:
    
    xray = read_xray('/kaggle/input/siim-covid19-detection/train/00086460a852/9e8302230c91/65761e66de9f.dcm')
    im = resize(xray, size=600)  
    study = '00086460a852' + '_study.png'
    im.save(os.path.join(save_dir, study))
    xray = read_xray('/kaggle/input/siim-covid19-detection/train/000c9c05fd14/e555410bd2cd/51759b5579bc.dcm')
    im = resize(xray, size=600)  
    study = '000c9c05fd14' + '_study.png'
    im.save(os.path.join(save_dir, study))
else:
    
    for dirname, _, filenames in tqdm(os.walk(f'/kaggle/input/siim-covid19-detection/{split}')):
        for file in filenames:
            # set keep_ratio=True to have original aspect ratio
            xray = read_xray(os.path.join(dirname, file))
            im = resize(xray, size=600)  
            study = dirname.split('/')[-2] + '_study.png'
            im.save(os.path.join(save_dir, study))

In [ ]:
split = 'test'

image_id = []
dim0 = []
dim1 = []
splits = []
save_dir = f'/kaggle/tmp/{split}/image/'
os.makedirs(save_dir, exist_ok=True)

if fast_sub:
    xray = read_xray('/kaggle/input/siim-covid19-detection/train/00086460a852/9e8302230c91/65761e66de9f.dcm')
    im = resize(xray, size=512)  
    im.save(os.path.join(save_dir,'65761e66de9f_image.png'))
    image_id.append('65761e66de9f.dcm'.replace('.dcm', ''))
    dim0.append(xray.shape[0])
    dim1.append(xray.shape[1])
    splits.append(split)
    xray = read_xray('/kaggle/input/siim-covid19-detection/train/000c9c05fd14/e555410bd2cd/51759b5579bc.dcm')
    im = resize(xray, size=512)  
    im.save(os.path.join(save_dir, '51759b5579bc_image.png'))
    image_id.append('51759b5579bc.dcm'.replace('.dcm', ''))
    dim0.append(xray.shape[0])
    dim1.append(xray.shape[1])
    splits.append(split)
else:
    for dirname, _, filenames in tqdm(os.walk(f'/kaggle/input/siim-covid19-detection/{split}')):
        for file in filenames:
            # set keep_ratio=True to have original aspect ratio
            xray = read_xray(os.path.join(dirname, file))
            im = resize(xray, size=512)  
            im.save(os.path.join(save_dir, file.replace('.dcm', '_image.png')))
            image_id.append(file.replace('.dcm', ''))
            dim0.append(xray.shape[0])
            dim1.append(xray.shape[1])
            splits.append(split)
meta = pd.DataFrame.from_dict({'image_id': image_id, 'dim0': dim0, 'dim1': dim1, 'split': splits})

# Inference data loader

In [ ]:
class SIIMCOVID19TestData(Dataset):
    def __init__(self):
        super().__init__()
        
        if fast_sub:
            self.submission_df = fast_df.copy()
        else:
            self.submission_df = pd.read_csv('/kaggle/input/siim-covid19-detection/sample_submission.csv')

        self.submission_df['StudyInstanceUID'] = self.submission_df['id'].apply(lambda x: x.replace('_study', ''))
        self.submission_df = self.submission_df[~self.submission_df['StudyInstanceUID'].str.endswith('_image')]
        
        if fast_sub:
            paths = ['/kaggle/input/siim-covid19-detection/train/00086460a852/9e8302230c91/65761e66de9f.dcm',
                     '/kaggle/input/siim-covid19-detection/train/000c9c05fd14/e555410bd2cd/51759b5579bc.dcm']
            
        else:
        
            TEST_DIR = "/kaggle/input/siim-covid19-detection/test/"

            # Make a path folder
            paths = []
            for instance_id in tqdm(self.submission_df['StudyInstanceUID']):
                paths.append(glob.glob(os.path.join(TEST_DIR, instance_id +"/*/*"))[0])

        
        self.submission_df['path'] = paths
        
        self.transform = transforms.Compose([
            transforms.ToTensor()
        ])

    def __len__(self):
        return len(self.submission_df)

    def __getitem__(self,item):
        image_id = self.submission_df['StudyInstanceUID'].values[item]
        
        image_path = self.submission_df['path'].values[item]
        image = dicom2array(image_path)
        image = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)
        image = cv2.resize(image, Config.image_size)
        
        image = albumentations.Compose([albumentations.Normalize()])(image=image)['image']
        
        image = torch.tensor(image)
        
        return {
                "x":image
                }

    

test_dataset = SIIMCOVID19TestData()

test_loader = DataLoader(test_dataset,
                        batch_size=32,
                        num_workers=8)

# Load models - Colab

In [ ]:
!mkdir -p /root/.cache/torch/hub/checkpoints/
!cp ../input/litmodels-bolts/resnet50-19c8e357.pth /root/.cache/torch/hub/checkpoints/
!cp ../input/litmodels-bolts/resnet18-5c106cde.pth /root/.cache/torch/hub/checkpoints/
!cp ../input/litmodels-bolts/efficientnet_b0_ra-3dd342df.pth /root/.cache/torch/hub/checkpoints/
!cp ../input/litmodels-bolts/efficientnet_b2_ra-bcdf34b7.pth /root/.cache/torch/hub/checkpoints/
!cp ../input/litmodels-bolts/efficientnet_b3_ra2-cf984f9c.pth /root/.cache/torch/hub/checkpoints/

In [ ]:
criterion = nn.CrossEntropyLoss()
softmax = nn.Softmax()

In [ ]:
ckpt_path1 = '../input/siimcovid19trainedmodelsstudyclassification/resnet50-siim-covid19-13-0.4999.ckpt'

resnet50_model = lit.LitResNet50.load_from_checkpoint(checkpoint_path = ckpt_path1)
resnet50_model = resnet50_model.to("cuda")
resnet50_model.eval()
resnet50_model.freeze()

In [ ]:
ckpt_path2 = '../input/siimcovid19trainedmodelsstudyclassification/resnet18-siim-covid19-33-0.4981.ckpt'

resnet18_model = lit.LitResNet18.load_from_checkpoint(checkpoint_path = ckpt_path2)
resnet18_model = resnet18_model.to("cuda")
resnet18_model.eval()
resnet18_model.freeze()

In [ ]:
ckpt_path3 = '../input/siimcovid19trainedmodelsstudyclassification/resnet18-siim-covid19-27-0.5138.ckpt'

resnet18_model2 = lit.LitResNet18_2.load_from_checkpoint(checkpoint_path = ckpt_path3)
resnet18_model2 = resnet18_model2.to("cuda")
resnet18_model2.eval()
resnet18_model2.freeze()

In [ ]:
ckpt_path4 = '../input/siimcovid19trainedmodelsstudyclassification/effnetb0-siim-covid19-4-0.5119.ckpt'

effnetb0_model = lit.LitEffNetB0.load_from_checkpoint(checkpoint_path = ckpt_path4)
effnetb0_model = effnetb0_model.to("cuda")
effnetb0_model.eval()
effnetb0_model.freeze()

In [ ]:
ckpt_path5 = '../input/siimcovid19trainedmodelsstudyclassification/effnetb2a-siim-covid19-14-0.4894.ckpt'

effnetb2a_model = lit.LitEffNetB2a.load_from_checkpoint(checkpoint_path = ckpt_path5)
effnetb2a_model = effnetb2a_model.to("cuda")
effnetb2a_model.eval()
effnetb2a_model.freeze()

In [ ]:
ckpt_path6 = '../input/siimcovid19trainedmodelsstudyclassification/effnetb3a-siim-covid19-26-0.4978.ckpt'

effnetb3a_model = lit.LitEffNetB3a.load_from_checkpoint(checkpoint_path = ckpt_path6)
effnetb3a_model = effnetb3a_model.to("cuda")
effnetb3a_model.eval()
effnetb3a_model.freeze()

# Inference - Colab

In [ ]:
def zerolistmaker(n):
    listofzeros = [0] * n
    return listofzeros

if fast_sub:
    num_preds = 2
else:
    num_preds = len(test_loader)*32

In [ ]:
preds_resnet50 = []

for ind, data in tqdm(enumerate(test_loader),total = len(test_loader)):
    y_hat = resnet50_model(data["x"].to("cuda"))
    y_hat = softmax(y_hat)
    preds_resnet50.extend(y_hat.cpu().detach().numpy().tolist())

# for i in range(num_preds):
#     preds_resnet50.append(zerolistmaker(4))

In [ ]:
preds_resnet18 = []

for ind, data in tqdm(enumerate(test_loader),total = len(test_loader)):
    y_hat = resnet18_model(data["x"].to("cuda"))
    y_hat = softmax(y_hat)
    preds_resnet18.extend(y_hat.cpu().detach().numpy().tolist())

# for i in range(num_preds):
#     preds_resnet18.append(zerolistmaker(4))

In [ ]:
preds_resnet18_2 = []

for ind, data in tqdm(enumerate(test_loader),total = len(test_loader)):
    y_hat = resnet18_model2(data["x"].to("cuda"))
    y_hat = softmax(y_hat)
    preds_resnet18_2.extend(y_hat.cpu().detach().numpy().tolist())

# for i in range(num_preds):
#     preds_resnet18_2.append(zerolistmaker(4))

In [ ]:
preds_effnetb0 = []

for ind, data in tqdm(enumerate(test_loader),total = len(test_loader)):
    y_hat = effnetb0_model(data["x"].to("cuda"))
    y_hat = softmax(y_hat)
    preds_effnetb0.extend(y_hat.cpu().detach().numpy().tolist())

# for i in range(num_preds):
#     preds_effnetb0.append(zerolistmaker(4))

In [ ]:
preds_effnetb2a = []

for ind, data in tqdm(enumerate(test_loader),total = len(test_loader)):
    y_hat = effnetb2a_model(data["x"].to("cuda"))
    y_hat = softmax(y_hat)
    preds_effnetb2a.extend(y_hat.cpu().detach().numpy().tolist())

# for i in range(num_preds):
#     preds_effnetb2a.append(zerolistmaker(4))

In [ ]:
preds_effnetb3a = []

for ind, data in tqdm(enumerate(test_loader),total = len(test_loader)):
    y_hat = effnetb3a_model(data["x"].to("cuda"))
    y_hat = softmax(y_hat)
    preds_effnetb3a.extend(y_hat.cpu().detach().numpy().tolist())

# for i in range(num_preds):
#     preds_effnetb3a.append(zerolistmaker(4))

# Inference - Efnb7

In [ ]:
import numpy as np 
import pandas as pd

if fast_sub:
    df = fast_df.copy()
else:
    df = pd.read_csv('/kaggle/input/siim-covid19-detection/sample_submission.csv')

id_laststr_list  = []
for i in range(df.shape[0]):
    id_laststr_list.append(df.loc[i,'id'][-1])
df['id_last_str'] = id_laststr_list

study_len = df[df['id_last_str'] == 'y'].shape[0]

In [ ]:
import os

import efficientnet.tfkeras as efn
import numpy as np
import pandas as pd
import tensorflow as tf

def auto_select_accelerator():
    try:
        tpu = tf.distribute.cluster_resolver.TPUClusterResolver()
        tf.config.experimental_connect_to_cluster(tpu)
        tf.tpu.experimental.initialize_tpu_system(tpu)
        strategy = tf.distribute.experimental.TPUStrategy(tpu)
        print("Running on TPU:", tpu.master())
    except ValueError:
        strategy = tf.distribute.get_strategy()
    print(f"Running on {strategy.num_replicas_in_sync} replicas")

    return strategy


def build_decoder(with_labels=True, target_size=(300, 300), ext='jpg'):
    def decode(path):
        file_bytes = tf.io.read_file(path)
        if ext == 'png':
            img = tf.image.decode_png(file_bytes, channels=3)
        elif ext in ['jpg', 'jpeg']:
            img = tf.image.decode_jpeg(file_bytes, channels=3)
        else:
            raise ValueError("Image extension not supported")

        img = tf.cast(img, tf.float32) / 255.0
        img = tf.image.resize(img, target_size)

        return img

    def decode_with_labels(path, label):
        return decode(path), label

    return decode_with_labels if with_labels else decode


def build_augmenter(with_labels=True):
    def augment(img):
        img = tf.image.random_flip_left_right(img)
        img = tf.image.random_flip_up_down(img)
        return img

    def augment_with_labels(img, label):
        return augment(img), label

    return augment_with_labels if with_labels else augment


def build_dataset(paths, labels=None, bsize=32, cache=True,
                  decode_fn=None, augment_fn=None,
                  augment=True, repeat=True, shuffle=1024, 
                  cache_dir=""):
    if cache_dir != "" and cache is True:
        os.makedirs(cache_dir, exist_ok=True)

    if decode_fn is None:
        decode_fn = build_decoder(labels is not None)

    if augment_fn is None:
        augment_fn = build_augmenter(labels is not None)

    AUTO = tf.data.experimental.AUTOTUNE
    slices = paths if labels is None else (paths, labels)

    dset = tf.data.Dataset.from_tensor_slices(slices)
    dset = dset.map(decode_fn, num_parallel_calls=AUTO)
    dset = dset.cache(cache_dir) if cache else dset
    dset = dset.map(augment_fn, num_parallel_calls=AUTO) if augment else dset
    dset = dset.repeat() if repeat else dset
    dset = dset.shuffle(shuffle) if shuffle else dset
    dset = dset.batch(bsize).prefetch(AUTO)

    return dset

#COMPETITION_NAME = "siim-cov19-test-img512-study-600"

In [ ]:
strategy = auto_select_accelerator()
BATCH_SIZE = strategy.num_replicas_in_sync * 16

IMSIZE = (224, 240, 260, 300, 380, 456, 528, 600, 512)

if fast_sub:
    sub_df = fast_df.copy()
else:
    sub_df = pd.read_csv('/kaggle/input/siim-covid19-detection/sample_submission.csv')

sub_df = sub_df[:study_len]
test_paths = f'/kaggle/tmp/{split}/study/' + sub_df['id'] +'.png'

sub_df['negative'] = 0
sub_df['typical'] = 0
sub_df['indeterminate'] = 0
sub_df['atypical'] = 0

In [ ]:
label_cols = sub_df.columns[2:]

test_decoder = build_decoder(with_labels=False, target_size=(IMSIZE[7], IMSIZE[7]), ext='png')
dtest = build_dataset(
    test_paths, bsize=BATCH_SIZE, repeat=False, 
    shuffle=False, augment=False, cache=False,
    decode_fn=test_decoder
)


In [ ]:
with strategy.scope():
    
    models = []
    
    models0 = tf.keras.models.load_model(
        '../input/siim-covid19-efnb7-train-study/model0.h5'
    )
    models1 = tf.keras.models.load_model(
        '../input/siim-covid19-efnb7-train-study/model1.h5'
    )
    models2 = tf.keras.models.load_model(
        '../input/siim-covid19-efnb7-train-study/model2.h5'
    )
    models3 = tf.keras.models.load_model(
        '../input/siim-covid19-efnb7-train-study/model3.h5'
    )
    models4 = tf.keras.models.load_model(
        '../input/siim-covid19-efnb7-train-study/model4.h5'
    )
    
    models.append(models0)
    models.append(models1)
    models.append(models2)
    models.append(models3)
    models.append(models4)


In [ ]:
preds_efnb7_1 = models0.predict(dtest, verbose=1)
preds_efnb7_2 = models1.predict(dtest, verbose=1)
preds_efnb7_3 = models2.predict(dtest, verbose=1)
preds_efnb7_4 = models3.predict(dtest, verbose=1)
preds_efnb7_5 = models4.predict(dtest, verbose=1)

In [ ]:
# preds_efnb7_1 = []
# preds_efnb7_2 = []
# preds_efnb7_3 = []
# preds_efnb7_4 = []
# preds_efnb7_5 = []

# for i in range(1214):
#     preds_efnb7_1.append(zerolistmaker(4))
#     preds_efnb7_2.append(zerolistmaker(4))
#     preds_efnb7_3.append(zerolistmaker(4))
#     preds_efnb7_4.append(zerolistmaker(4))
#     preds_efnb7_5.append(zerolistmaker(4))

# Study level predictions

In [ ]:
wt_restnet50 = 0.05
wt_restnet18 = 0.05
wt_restnet18_2 = 0.05
wt_effnetb0 = 0.05
wt_effnetb2a = 0.05
wt_effnetb3a = 0.05
wt_efnb7 = 0.7

wt_efnb7_1 = 0.1
wt_efnb7_2 = 0.1
wt_efnb7_3 = 0.1
wt_efnb7_4 = 0.1
wt_efnb7_5 = 0.6

print(wt_restnet50 + wt_restnet18 + wt_restnet18_2 + wt_effnetb0 + wt_effnetb2a + wt_effnetb3a + wt_efnb7)
print(wt_efnb7_1 + wt_efnb7_2 + wt_efnb7_3 + wt_efnb7_4 + wt_efnb7_5) 

In [ ]:
submission_df = pd.read_csv("/kaggle/input/siim-covid19-detection/sample_submission.csv")

study_submission_df = submission_df[submission_df['id'].str.endswith('_study')]
image_submission_df = submission_df[submission_df['id'].str.endswith('_image')]

for ind, pred in tqdm(enumerate(preds_resnet18_2), total=len(preds_resnet18_2)):

    finalpred = []

    finalpred.append((wt_restnet50*pred[0] + wt_restnet18*preds_resnet18[ind][0] + wt_restnet18_2*preds_resnet18_2[ind][0] + \
                          wt_effnetb0*preds_effnetb0[ind][0] + wt_effnetb2a*preds_effnetb2a[ind][0] + wt_effnetb3a*preds_effnetb3a[ind][0] + \
                          wt_efnb7*((wt_efnb7_1*preds_efnb7_1[ind][0] + wt_efnb7_2*preds_efnb7_2[ind][0] + wt_efnb7_3*preds_efnb7_3[ind][0] + wt_efnb7_4*preds_efnb7_4[ind][0] + wt_efnb7_5*preds_efnb7_5[ind][0]))))

    finalpred.append((wt_restnet50*pred[1] + wt_restnet18*preds_resnet18[ind][1] + wt_restnet18_2*preds_resnet18_2[ind][1] + \
                      wt_effnetb0*preds_effnetb0[ind][1] + wt_effnetb2a*preds_effnetb2a[ind][1] + wt_effnetb3a*preds_effnetb3a[ind][1] + \
                      wt_efnb7*((wt_efnb7_1*preds_efnb7_1[ind][1] + wt_efnb7_2*preds_efnb7_2[ind][1] + wt_efnb7_3*preds_efnb7_3[ind][1] + wt_efnb7_4*preds_efnb7_4[ind][1] + wt_efnb7_5*preds_efnb7_5[ind][1]))))

    finalpred.append((wt_restnet50*pred[2] + wt_restnet18*preds_resnet18[ind][2] + wt_restnet18_2*preds_resnet18_2[ind][2] + \
                      wt_effnetb0*preds_effnetb0[ind][2] + wt_effnetb2a*preds_effnetb2a[ind][2] + wt_effnetb3a*preds_effnetb3a[ind][2] + \
                      wt_efnb7*((wt_efnb7_1*preds_efnb7_1[ind][2] + wt_efnb7_2*preds_efnb7_2[ind][2] + wt_efnb7_3*preds_efnb7_3[ind][2] + wt_efnb7_4*preds_efnb7_4[ind][2] + wt_efnb7_5*preds_efnb7_5[ind][2]))))

    finalpred.append((wt_restnet50*pred[3] + wt_restnet18*preds_resnet18[ind][3] + wt_restnet18_2*preds_resnet18_2[ind][3] + \
                      wt_effnetb0*preds_effnetb0[ind][3] + wt_effnetb2a*preds_effnetb2a[ind][3] + wt_effnetb3a*preds_effnetb3a[ind][3] + \
                      wt_efnb7*((wt_efnb7_1*preds_efnb7_1[ind][3] + wt_efnb7_2*preds_efnb7_2[ind][3] + wt_efnb7_3*preds_efnb7_3[ind][3] + wt_efnb7_4*preds_efnb7_4[ind][3] + wt_efnb7_5*preds_efnb7_5[ind][3]))))

    negative, typical, indeterminate, atypical = str(finalpred[0]),str(finalpred[1]),str(finalpred[2]),str(finalpred[3])
#     negative, typical, indeterminate, atypical = str(preds_effnetb0[ind][0]),str(preds_effnetb0[ind][1]),str(preds_effnetb0[ind][2]),str(preds_effnetb0[ind][3])
    study_submission_df.loc[ind,'PredictionString'] = f'negative {negative} 0 0 1 1 typical {typical} 0 0 1 1 indeterminate {indeterminate} 0 0 1 1 atypical {atypical} 0 0 1 1'

    
# submission_df = pd.concat([study_submission_df, image_submission_df],axis=0, ignore_index=True)

print(study_submission_df.shape)

# 2 class prediction (Opacity/ None)

In [ ]:
if fast_sub:
    sub_df = fast_df.copy()
else:
    sub_df = pd.read_csv('/kaggle/input/siim-covid19-detection/sample_submission.csv')
sub_df = sub_df[study_len:]
test_paths = f'/kaggle/tmp/{split}/image/' + sub_df['id'] +'.png'
sub_df['none'] = 0

label_cols = sub_df.columns[2]

test_decoder = build_decoder(with_labels=False, target_size=(IMSIZE[8], IMSIZE[8]), ext='png')
dtest = build_dataset(
    test_paths, bsize=BATCH_SIZE, repeat=False, 
    shuffle=False, augment=False, cache=False,
    decode_fn=test_decoder
)

with strategy.scope():
    
    models = []
    
    models0 = tf.keras.models.load_model(
        '/kaggle/input/siim-covid19-efnb7-train-fold0-5-2class/model0.h5'
    )
    models1 = tf.keras.models.load_model(
        '/kaggle/input/siim-covid19-efnb7-train-fold0-5-2class/model1.h5'
    )
    models2 = tf.keras.models.load_model(
        '/kaggle/input/siim-covid19-efnb7-train-fold0-5-2class/model2.h5'
    )
    models3 = tf.keras.models.load_model(
        '/kaggle/input/siim-covid19-efnb7-train-fold0-5-2class/model3.h5'
    )
    models4 = tf.keras.models.load_model(
        '/kaggle/input/siim-covid19-efnb7-train-fold0-5-2class/model4.h5'
    )
    
    models.append(models0)
    models.append(models1)
    models.append(models2)
    models.append(models3)
    models.append(models4)

weights = {
    0: 1,
    1: 1,
    2: 1,
    3: 1,
    4: 3
}

weights_sum = sum(weights.values())
weights = {k: v/weights_sum for k, v in weights.items()}

predictions = [model.predict(dtest, verbose=1) for model in models]
for i, pred in enumerate(predictions):
    predictions[i] = weights[i] * pred
    
sub_df[label_cols] = sum(predictions)
#sub_df[label_cols] = sum([model.predict(dtest, verbose=1) for model in models]) / len(models)
df_2class = sub_df.reset_index(drop=True)

In [ ]:
df_2class.head()

In [ ]:
del models
del models0, models1, models2, models3, models4

In [ ]:
from numba import cuda
import torch
cuda.select_device(0)
cuda.close()
cuda.select_device(0)

# Yolov5: Image level Object detection

In [ ]:
import numpy as np, pandas as pd
from glob import glob
import shutil, os
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupKFold
from tqdm.notebook import tqdm
import seaborn as sns
import torch

In [ ]:
meta = meta[meta['split'] == 'test']
if fast_sub:
    test_df = fast_df.copy()
else:
    test_df = pd.read_csv('/kaggle/input/siim-covid19-detection/sample_submission.csv')
test_df = df[study_len:].reset_index(drop=True) 
meta['image_id'] = meta['image_id'] + '_image'
meta.columns = ['id', 'dim0', 'dim1', 'split']
test_df = pd.merge(test_df, meta, on = 'id', how = 'left')

In [ ]:
dim = 512 #1024, 256, 'original'
test_dir = f'/kaggle/tmp/{split}/image'
#weights_dir = '/kaggle/input/siim-cov19-yolov5-train/yolov5/runs/train/exp/weights/best.pt'
weights_dir = '/kaggle/input/weights-of-yolov5-150-epochs/best.pt'

shutil.copytree('/kaggle/input/yolov5-repo/yolov5-master', '/kaggle/working/yolov5')
os.chdir('/kaggle/working/yolov5') # install dependencies
!git pull

import torch
#from IPython.display import Image, clear_output  # to display images

#clear_output()
#print('Setup complete. Using torch %s %s' % (torch.__version__, torch.cuda.get_device_properties(0) if torch.cuda.is_available() else 'CPU'))


!python detect.py --weights {weights_dir}\
--img 512\
--conf 0.005\
--iou 0.6\
# --max-det 25\
--augment\
--source {test_dir}\
--save-txt --save-conf --exist-ok
def yolo2voc(image_height, image_width, bboxes):
    """
    yolo => [xmid, ymid, w, h] (normalized)
    voc  => [x1, y1, x2, y1]

    """ 
    bboxes = bboxes.copy().astype(float) # otherwise all value will be 0 as voc_pascal dtype is np.int

    bboxes[..., [0, 2]] = bboxes[..., [0, 2]]* image_width
    bboxes[..., [1, 3]] = bboxes[..., [1, 3]]* image_height

    bboxes[..., [0, 1]] = bboxes[..., [0, 1]] - bboxes[..., [2, 3]]/2
    bboxes[..., [2, 3]] = bboxes[..., [0, 1]] + bboxes[..., [2, 3]]

    return bboxes
image_ids = []
PredictionStrings = []

for file_path in tqdm(glob('runs/detect/exp/labels/*.txt')):
    image_id = file_path.split('/')[-1].split('.')[0]
    w, h = test_df.loc[test_df.id==image_id,['dim1', 'dim0']].values[0]
    f = open(file_path, 'r')
    data = np.array(f.read().replace('\n', ' ').strip().split(' ')).astype(np.float32).reshape(-1, 6)
    data = data[:, [0, 5, 1, 2, 3, 4]]
    bboxes = list(np.round(np.concatenate((data[:, :2], np.round(yolo2voc(h, w, data[:, 2:]))), axis =1).reshape(-1), 12).astype(str))
    for idx in range(len(bboxes)):
        bboxes[idx] = str(int(float(bboxes[idx]))) if idx%6!=1 else bboxes[idx]
    image_ids.append(image_id)
    PredictionStrings.append(' '.join(bboxes))


pred_df = pd.DataFrame({'id':image_ids,
                        'PredictionString':PredictionStrings})

In [ ]:
test_df = test_df.drop(['PredictionString'], axis=1)
sub_df = pd.merge(test_df, pred_df, on = 'id', how = 'left').fillna("none 1 0 0 1 1")
sub_df = sub_df[['id', 'PredictionString']]
for i in range(sub_df.shape[0]):
    if sub_df.loc[i,'PredictionString'] == "none 1 0 0 1 1":
        continue
    sub_df_split = sub_df.loc[i,'PredictionString'].split()
    sub_df_list = []
    for j in range(int(len(sub_df_split) / 6)):
        sub_df_list.append('opacity')
        sub_df_list.append(sub_df_split[6 * j + 1])
        sub_df_list.append(sub_df_split[6 * j + 2])
        sub_df_list.append(sub_df_split[6 * j + 3])
        sub_df_list.append(sub_df_split[6 * j + 4])
        sub_df_list.append(sub_df_split[6 * j + 5])
    sub_df.loc[i,'PredictionString'] = ' '.join(sub_df_list)
sub_df['none'] = df_2class['none'] 
for i in range(sub_df.shape[0]):
    if sub_df.loc[i,'PredictionString'] != 'none 1 0 0 1 1':
        sub_df.loc[i,'PredictionString'] = sub_df.loc[i,'PredictionString'] + ' none ' + str(sub_df.loc[i,'none']) + ' 0 0 1 1'
sub_df = sub_df[['id', 'PredictionString']]   

In [ ]:
sub_df.shape

# Final submission file

In [ ]:
submission_df = study_submission_df.append(sub_df).reset_index(drop=True)
submission_df.to_csv('/kaggle/working/submission.csv',index = False)  
shutil.rmtree('/kaggle/working/yolov5')

In [ ]:
submission_df.shape

In [ ]:
submission_df.head()

In [ ]:
submission_df.tail()